# 04 — Classifier Training

Train a Random Forest on ECFP4 fingerprints. Evaluate with stratified 5-fold
cross-validation and a held-out test set.

**Input:** `features_X.npy`, `features_y.npy`
**Output:** `../data/processed/models/combined_model.joblib`

In [ ]:
import sys, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from pathlib import Path

sys.path.insert(0, '..')
from src.models import (
    build_classifier, cross_validate, train_and_evaluate,
    save_model, get_feature_importance,
)

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

Path('../data/processed/models').mkdir(parents=True, exist_ok=True)

X = np.load('../data/processed/features_X.npy')
y = np.load('../data/processed/features_y.npy')

print(f"Feature matrix : {X.shape}")
print(f"Active         : {y.sum()} ({y.mean():.1%})")
print(f"Inactive       : {(y==0).sum()}")

## Train / test split

In [ ]:
cfg = config['models']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=cfg['test_size'], stratify=y, random_state=cfg['random_state'],
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## Stratified cross-validation

Estimate generalisation before touching the test set.
`class_weight='balanced'` corrects for the active/inactive imbalance.

In [ ]:
model = build_classifier(model_type=cfg['type'], n_estimators=cfg['n_estimators'])
cv = cross_validate(model, X_train, y_train, cv_folds=cfg['cv_folds'])

print(f"CV  ROC-AUC : {cv['roc_auc_mean']:.3f} ± {cv['roc_auc_std']:.3f}")
print(f"CV  PR-AUC  : {cv['pr_auc_mean']:.3f} ± {cv['pr_auc_std']:.3f}")

## Final model training and held-out test evaluation

In [ ]:
metrics = train_and_evaluate(model, X_train, y_train, X_test, y_test)

print(f"Test ROC-AUC : {metrics['roc_auc']:.3f}")
print(f"Test PR-AUC  : {metrics['pr_auc']:.3f}")
print()
print(metrics['classification_report'])

model_path = '../data/processed/models/combined_model.joblib'
save_model(model, model_path)

## ROC and Precision–Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = metrics['roc_curve']
axes[0].plot(fpr, tpr, lw=2, color='steelblue', label=f"AUC = {metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

prec, rec, _ = metrics['pr_curve']
axes[1].plot(rec, prec, lw=2, color='darkorange', label=f"AUC = {metrics['pr_auc']:.3f}")
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision–Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/model_evaluation_curves.png', dpi=150)
plt.show()

## Feature importance (top 20 Morgan bits)

Each bar is a fingerprint bit. High importance = that circular substructure
pattern strongly separates active from inactive compounds.

In [ ]:
importances = get_feature_importance(model, top_n=20)

fig, ax = plt.subplots(figsize=(8, 5))
importances.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Feature importance')
ax.set_ylabel('ECFP4 bit index')
ax.set_title('Top 20 most discriminative substructure bits')
plt.tight_layout()
plt.savefig('../data/processed/feature_importance.png', dpi=150)
plt.show()